In [1]:
import json
import pandas as pd
import numpy as np
import os
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm 
from PIL import Image
from diffusers import StableDiffusionPipeline
import torch

In [2]:
model_id = "sd-legacy/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

c:\Users\DaisLabTSB\.conda\envs\Giando\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DaisLabTSB\.cache\huggingface\hub\models--sd-legacy--stable-diffusion-v1-5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

text_encoder/config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer/special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

unet/config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
def json_to_dataframe(json_data):
    """
    Converts JSON data (string or dictionary/list) to a Pandas DataFrame.

    Args:
        json_data (str or dict/list): The JSON data as a string, 
                                       a dictionary (for a single entry), or
                                       a list of dictionaries.

    Returns:
        pandas.DataFrame: A DataFrame containing the data, 
                         or None if an error occurs.
    """
    try:
        if isinstance(json_data, str):
            data = json.loads(json_data) # Parse JSON string if it's a string
        elif isinstance(json_data, (dict, list)): # If it's already parsed
            data = json_data
        else:
            print("Invalid input: Please provide a JSON string or a list/dictionary.")
            return None

        if isinstance(data, list):
            df = pd.DataFrame(data)
            return df
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
            return df
        else:
            print("JSON data does not contain a list or dictionary of entries.")
            return None

    except json.JSONDecodeError:
        print("Error: Invalid JSON format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [4]:
# Open and read the JSON file
with open('../clip_embedding_aimagelab.json', 'r') as file:
    data_aimagelab = json.load(file)

# Open and read the JSON file
with open('../clip_embedding_CoPro.json', 'r') as file:
    data_CoPro = json.load(file)

In [5]:
df_aimagelab = json_to_dataframe(data_aimagelab)
df_CoPro = json_to_dataframe(data_CoPro)

In [6]:
df_CoPro["incremental_id"] = range(1, len(df_CoPro) + 1)

In [7]:
# Definisci la cartella di salvataggio
output_folder = "stable-diffusion-v1-5"
os.makedirs(output_folder, exist_ok=True)

In [8]:
# prompt = "a photo of an astronaut riding a horse on mars"
# image = pipe(prompt).images[0]  
    
# image.save("astronaut_rides_horse.png")

In [9]:
# Funzione per generare immagini in batch e aggiornare il DataFrame
def generate_images_and_update_df(df, df_name, batch_size=4):
    """Genera immagini in batch e aggiorna il DataFrame con gli ID delle immagini."""
    
    path = os.path.join(output_folder, df_name)
    os.makedirs(path, exist_ok=True)
    
    image_ids = []
    prompts = df["prompt"].tolist()
    
    # Elaborazione in batch
    for i in tqdm(range(0, len(prompts), batch_size), total=len(prompts)//batch_size, desc=f"Processing {df_name}"):
        batch_prompts = prompts[i:i+batch_size]
        
        try:
            images = pipe(batch_prompts).images  # Genera batch di immagini
            for j, image in enumerate(images):
                image_id = f"{df_name}_{i+j}.png"
                image_path = os.path.join(path, image_id)
                image.save(image_path)
                image_ids.append(image_id)
        except Exception as e:
            print(f"Errore nel batch {i}: {e}")
            image_ids.extend([None] * len(batch_prompts))  # Segna errori
        
    df["image_id"] = image_ids
    return df


In [10]:
# Processa entrambi i DataFrame
df_aimagelab = generate_images_and_update_df(df_aimagelab, "aimagelab")
df_CoPro = generate_images_and_update_df(df_CoPro, "CoPro")

# Salva i DataFrame aggiornati
df_aimagelab.to_csv(output_folder+"df_aimagelab_updated.csv", index=False)
df_CoPro.to_csv(output_folder+"df_CoPro_updated.csv", index=False)

Processing aimagelab: 0it [00:00, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

C:\Users\giand\AppData\Local\Temp\ipykernel_8748\2603620249.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["image_id"] = image_ids
